# RNA3D Phase 1 — Template Scarcity & Training-Overlap Evidence Inventory

**CPU-only • no inference • no GPU • no GitHub writes • previous experiments preserved.**

This is an **evidence inventory and gap audit**, not a complete structure-template search or training-set membership screen. The notebook joins the finalized native-chain audit with saved MSA search results and the existing native-mapped evaluation. It separates **PDB deposition**, **first revision date** and **verified release date** (the latter is *not* established from existing audit outputs), and labels whether training-data exclusion is *verified*, *plausible by chronology only*, or *unresolved*.

**Interpretation guardrails:** MSA depth 1 does not prove no structural templates. Low sequence similarity cannot establish low structural similarity. A post-cutoff deposited target does not exclude homologous training structures. Temporal claims must use the actual checkpoint's training snapshot, not a paper publication date.


## 1. Connect Google Drive — CPU runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import csv, json, hashlib, math, shutil, re
from pathlib import Path
from collections import Counter
from datetime import datetime, date, timezone
from statistics import mean


## 2. Exact pinned input folders and *new* output ID

In [ ]:
ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
AUDIT=ROOT/'Evaluations/pilot10_native_chain_provenance_20261113_v1'
MSA=ROOT/'MSA/search_results/pilot10_msa_availability.csv'
THREE=ROOT/'RhoFold/runs/rhofoldplus_pilot10_single_relax1000_20261111_v1/evaluations/rhofoldplus_threeway_usalign20260527_20261112_v1'
RUN_ID='pilot10_template_leakage_evidence_inventory_20261113_v1'
OUT=ROOT/'Evaluations'/RUN_ID
STAGE=Path('/content/rna3d_template_leakage_inventory_stage_20261113')
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B',
         '9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
assert len(set(TARGETS))==10
for p in [AUDIT,THREE,MSA]: assert p.exists(),f'Missing input: {p}'
assert not OUT.exists(),f'Immutable output exists; inspect it rather than overwrite: {OUT}'
assert not STAGE.exists(),f'Local staging already exists; restart a fresh runtime: {STAGE}'
print('Inputs verified. No GPU required. New output:', OUT)


## 3. Read-only evidence integrity and join

In [ ]:
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for z in iter(lambda:f.read(1048576),b''):h.update(z)
    return h.hexdigest()
def csv_rows(p):
    with Path(p).open(newline='') as f:return list(csv.DictReader(f))
def verify_final(dir_path):
    marker=dir_path/'FINALIZED.json'
    assert marker.exists(),f'Incomplete run: {dir_path}'
    j=json.loads(marker.read_text())
    assert j['status']=='complete',f'Run not complete: {dir_path}'
    for name,expected in j['files'].items():
        p=dir_path/name
        assert p.exists() and sha256(p)==expected,f'Source modified/missing: {p}'
    return j
native_final=verify_final(AUDIT)
three_final=verify_final(THREE)
native={r['target_id']:r for r in csv_rows(AUDIT/'native_chain_audit.csv')}
msa={r['target_id']:r for r in csv_rows(MSA)}
paired={r['target_id']:r for r in csv_rows(THREE/'three_model_paired.csv')}
assert set(native)==set(msa)==set(paired)==set(TARGETS)
assert len(native)==len(msa)==len(paired)==10
print('INTEGRITY PASS: two archived finalized runs and 10/10 joined target IDs')
print('MSA table SHA256:',sha256(MSA))


## 4. Cutoff interpretation: literature-derived context, NOT a verified checkpoint manifest

The **DRfold2 research article** states its **3D structural training set contained structures released before 2024** and test-set RNA sequences were screened against the training set at an 80% sequence-identity ceiling. This describes the authors' published protocol; it does not prove that **our specific pilot targets** lack homologous structures or that all components of our pinned checkpoint were frozen at that time.

The **RhoFold+ article** describes BGSU representative RNA sets **version 2022-04-13**, clustered at 80% similarity. This is a source-set snapshot and **not automatically a comprehensive cutoff for every pretrained component**. NuFold's exact checkpoint training-set exclusion has **not yet been independently verified**.

Source references to inspect manually:
- DRfold2: https://journals.plos.org/plosbiology/article?id=10.1371/journal.pbio.3003659
- RhoFold+: https://doi.org/10.1038/s41592-024-02487-0
- NuFold upstream: https://github.com/kiharalab/NuFold

**No model is labeled LEAK-FREE here.** 


In [ ]:
MODEL_EVIDENCE={
 'NuFold':{'status':'TRAINING_SET_NOT_VERIFIED',
           'reference':'https://github.com/kiharalab/NuFold',
           'verified_checkpoint_cutoff_date':None,
           'notes':'Exact checkpoint training chains, sequence overlap, and structure leakage require audit.'},
 'DRfold2':{'status':'PUBLISHED_PROTOCOL_ONLY',
           'reference':'https://journals.plos.org/plosbiology/article?id=10.1371/journal.pbio.3003659',
           'published_structure_release_before':'2024-01-01',
           'verified_checkpoint_cutoff_date':None,
           'notes':'Paper claims structures released before 2024 and a training/test identity exclusion; not independently checked against these 10 targets.'},
 'RhoFold+':{'status':'PUBLISHED_SOURCE_SET_ONLY',
           'reference':'https://doi.org/10.1038/s41592-024-02487-0',
           'published_bgsu_set_version':'2022-04-13',
           'verified_checkpoint_cutoff_date':None,
           'notes':'BGSU snapshot date is not a universal freeze date for all RhoFold+/RNA-FM training components.'}}
print('Literature context loaded for manual validation; no training exclusions auto-certified.')


## 5. Per-target screening matrix (strict separation of evidence and unknowns)

In [ ]:
def fnum(x):
    try:return float(x)
    except (TypeError,ValueError):return float('nan')
rows=[]
for t in TARGETS:
    a=native[t];m=msa[t];p=paired[t]
    assert a['pdb_id'].lower()==t.split('_',1)[0]
    assert a['status'] in {'PASS','REVIEW'}
    dep=a['initial_deposition_date']
    rev=a['earliest_revision_date']
    assert re.fullmatch(r'\d{4}-\d\d-\d\d',dep) and re.fullmatch(r'\d{4}-\d\d-\d\d',rev)
    depth=int(m['final_msa_depth']);hit_rfam=int(m['rfam_hits']);hit_rc=int(m['rnacentral_hits'])
    assert depth>=1 and hit_rfam>=0 and hit_rc>=0
    frac=fnum(a['native_fraction']);assert 0<frac<=1
    # The published training-structure criterion is release date, *not* deposition date.
    # Deposit date alone cannot certify no structure or homolog was present in training.
    row={'target_id':t,'pdb_id':a['pdb_id'],
         'author_chain':a['author_chain'],'label_asym_id':a['label_asym_id'],
         'native_audit_status':a['status'],
         'deposition_date':dep,'first_revision_date_not_proven_release':rev,
         'official_pdb_release_date_verified':'NOT_VERIFIED',
         'fasta_length':int(a['fasta_length']),'native_residue_count':int(a['native_residue_count']),
         'native_fraction':frac,'msa_depth':depth,'rnacentral_hits':hit_rc,'rfam_hits':hit_rfam,
         'msa_search_interpretation':'SOME_HOMOLOGS_FOUND' if depth>1 else 'NONE_FOUND_IN_THIS_SEARCH',
         'drfold2_literature_training_release_pre2024':'PUBLISHED_NOT_VERIFIED_ON_PINNED_CHECKPOINT',
         'rhofold_literature_BGSU_2022':'PUBLISHED_NOT_VERIFIED_ON_PINNED_CHECKPOINT',
         'nufold_training_overlap_status':'UNKNOWN',
         'direct_training_structure_membership':'NOT_TESTED',
         'training_sequence_identity_screen':'NOT_TESTED',
         'structure_template_similarity_screen':'NOT_TESTED',
         'experimental_reference_confidence':'REVIEW_MISSING_NATIVE_COORDINATES' if frac<.8 else 'PASS_NATIVE_CHAIN_MAPPING',
         'nufold_tm_score':float(p['nufold_rank1_tm_score']),
         'drfold2_tm_score':float(p['drfold2_tm_score']),
         'rhofoldplus_tm_score':float(p['rhofoldplus_single_tm_score']),
         'overall_template_scarcity_status':'UNDETERMINED'}
    rows.append(row)
print('EVIDENCE MATRIX COMPLETE:',len(rows),'/10')
for r in rows:
    print(f"{r['target_id']:9s} native={r['native_fraction']:.1%} MSA_depth={r['msa_depth']} deposition={r['deposition_date']} template_scarcity={r['overall_template_scarcity_status']}")
assert sum(r['msa_depth']==1 for r in rows)==9
assert sum(r['native_fraction']<.8 for r in rows)==2


## 6. Priority decisions / gap report

In [ ]:
high_native=[r for r in rows if r['native_fraction']>=.8]
review=[r for r in rows if r['native_fraction']<.8]
# Conservative preregistered screening priorities: preserve all target IDs, do not discard older runs.
followup={
 'all_targets':TARGETS,
 'ready_for_template_screening':[r['target_id'] for r in high_native],
 'native_coverage_review_first':[r['target_id'] for r in review],
 'required_to_claim_template_scarcity':[
  'Obtain actual RhoFold+/NuFold/DRfold2 checkpoint training-chain lists and component-specific cutoffs.',
  'Retrieve official PDB release dates; do not substitute initial deposition or earliest revision date.',
  'Conduct reproducible sequence identity/coverage screening against training structures and historical PDB.',
  'Conduct 3D structural similarity screening and document source-database snapshot, thresholds, coverage, and exclusion criteria.',
  'Distinguish direct target overlap, homologous-template availability, and possible language-model pretraining overlap.',
  'Rerun any target subset selection according to predefined exclusions only; report both full exploratory and filtered sets.'],
 'caveats':['No leakage clearance proven','MSA depth is not structural template scarcity','RhoFold+ run is single-sequence quick reference','10-target exploratory sample only']}
print('High native-coordinate cohort:',', '.join(followup['ready_for_template_screening']))
print('Needs native-coverage caution:',', '.join(followup['native_coverage_review_first']))
print('Training overlap/template status: UNKNOWN for ALL 10 (no unsupported claims).')


## 7. Save a new immutable evidence inventory to Google Drive

In [ ]:
STAGE.mkdir(parents=True,exist_ok=False)
with (STAGE/'target_evidence_inventory.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys()),lineterminator='\n')
    w.writeheader();w.writerows(rows)
summary={'run_id':RUN_ID,'status':'complete_evidence_inventory_not_full_leakage_screen',
         'created_utc':datetime.now(timezone.utc).isoformat(),
         'count_targets':len(rows),'native_mapping_pass':sum(r['native_audit_status']=='PASS' for r in rows),
         'native_mapping_review':sum(r['native_audit_status']=='REVIEW' for r in rows),
         'msa_depth_one_targets':sum(r['msa_depth']==1 for r in rows),
         'actual_training_set_membership_verified':0,
         'template_scarcity_verified':0,
         'source_msa_sha256':sha256(MSA),
         'source_native_finalized_sha256':sha256(AUDIT/'FINALIZED.json'),
         'source_threeway_finalized_sha256':sha256(THREE/'FINALIZED.json'),
         'model_evidence':MODEL_EVIDENCE,
         'followup':followup}
(STAGE/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
assert not OUT.exists()
OUT.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(STAGE,OUT)
required=['target_evidence_inventory.csv','summary.json']
assert all(sha256(STAGE/f)==sha256(OUT/f) for f in required)
(OUT/'FINALIZED.json').write_text(json.dumps({
 'status':'complete','run_id':RUN_ID,'files':{f:sha256(OUT/f) for f in required},
 'created_utc':datetime.now(timezone.utc).isoformat()},indent=2)+'\n')
assert len(csv_rows(OUT/'target_evidence_inventory.csv'))==10
assert json.loads((OUT/'FINALIZED.json').read_text())['status']=='complete'
print('EVIDENCE INVENTORY SAVED AND VERIFIED:',OUT)
print('NEXT ACTION: obtain training-set snapshots and conduct real historical sequence/structure template screening.')
print('NO MODEL INFERENCE; NO GPU; NO GITHUB WRITE.')
